<a href="https://colab.research.google.com/github/MelB18/EasyQuanten/blob/main/Stundenplan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install qiskit-optimization

In [2]:
!pip install qiskit-algorithms

In [3]:
from qiskit_optimization import QuadraticProgram

# 1. Problem instanziieren
qp = QuadraticProgram(name="mein_optimierungsproblem")

# 2. Variablen konfigurieren
x = qp.binary_var(name="x")                          # Binär (0 oder 1)
y = qp.integer_var(lowerbound=0, upperbound=10, name="y") # Ganzzahl mit Grenzen
z = qp.continuous_var(lowerbound=-2.5, name="z")     # Kontinuierlich

# 3. Zielfunktion konfigurieren (Maximierung von: 3x + 2y - 5x*y)
qp.maximize(
    linear={"x": 3, "y": 2},
    quadratic={("x", "y"): -5}
)

# 4. Nebenbedingungen konfigurieren
# Lineare Bedingung: x + 2y <= 10
qp.linear_constraint(linear={"x": 1, "y": 2}, sense="<=", rhs=10, name="lin_cond")

# Quadratische Bedingung: x*y + z >= 2
qp.quadratic_constraint(linear={"z": 1}, quadratic={("x", "y"): 1}, sense=">=", rhs=2, name="quad_cond")

# Problem-Struktur anzeigen
print(qp.export_as_lp_string())

\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: mein_optimierungsproblem

Maximize
 obj: 3 x + 2 y + [ - 10 x*y ]/2
Subject To
 lin_cond: x + 2 y <= 10
 quad_cond: [ x*y ] + z >= 2

Bounds
 0 <= x <= 1
       y <= 10
 -2.500000000000 <= z

Binaries
 x

Generals
 y
End



/tmp/ipykernel_1475/2238811342.py:25: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print(qp.export_as_lp_string())


In [19]:
from qiskit_optimization import QuadraticProgram

# 1. Problem instanziieren
qp = QuadraticProgram(name="Stundenplan_Optimierung")

# 2. Parameter definieren
zeitslots = ["Stunde_1", "Stunde_2", "Stunde_3"]
klassen = ["Klasse_A", "Klasse_B"]
lehrer = ["Herr_Mueller", "Frau_Schmidt"]

# 3. Binäre Variablen dynamisch erstellen
# Format der Variable: x_Zeitslot_Klasse_Lehrer
print("Binäre Variablen:")
for t in zeitslots:
    for c in klassen:
        for tchr in lehrer:
            var_name = f"x_{t}_{c}_{tchr}"
            qp.binary_var(name=var_name)
            print(var_name)

# 4. Nebenbedingung 1: Eine Klasse hat pro Stunde max. einen Lehrer
# Für jedes t und jedes c darf die Summe über alle Lehrer maximal 1 sein.
print("Nebenbedingung 1: Eine Klasse hat pro Stunde max. einen Lehrer")
for t in zeitslots:
    for c in klassen:
        constraint_dict = {}
        for tchr in lehrer:
            var_name = f"x_{t}_{c}_{tchr}"
            constraint_dict[var_name] = 1

        qp.linear_constraint(
            linear=constraint_dict,
            sense="<=",
            rhs=1,
            name=f"Constraint_Klasse_{c}_zur_{t}"
        )
        print(constraint_dict)

# 5. Nebenbedingung 2: Ein Lehrer unterrichtet pro Stunde max. eine Klasse
# Für jedes t und jeden tchr darf die Summe über alle Klassen maximal 1 sein.
print("Nebenbedingung 2: Ein Lehrer unterrichtet pro Stunde max. eine Klasse")
for t in zeitslots:
    for tchr in lehrer:
        constraint_dict = {}
        for c in klassen:
            var_name = f"x_{t}_{c}_{tchr}"
            constraint_dict[var_name] = 1

        qp.linear_constraint(
            linear=constraint_dict,
            sense="<=",
            rhs=1,
            name=f"Constraint_Lehrer_{tchr}_zur_{t}"
        )
        print(constraint_dict)

# Anforderung: Herr Müller muss insgesamt exakt 2 Stunden in Klasse A unterrichten
anforderung_stunden = 2
constraint_dict = {}

for t in zeitslots:
    var_name = f"x_{t}_Klasse_A_Herr_Mueller"
    constraint_dict[var_name] = 1

qp.linear_constraint(
    linear=constraint_dict,
    sense="==",  # Exakt gleich
    rhs=anforderung_stunden,
    name="Anforderung_Mueller_KlasseA_2Std"
)
print(constraint_dict)


# Anforderung: Herr Müller kann in Stunde 1 KEINE Klasse unterrichten (Verfügbarkeit)
for c in klassen:
    var_name = f"x_Stunde_1_{c}_Herr_Mueller"
    # Wir setzen diese Variable fix auf 0
    qp.linear_constraint(
        linear={var_name: 1},
        sense="==",
        rhs=0,
        name=f"Mueller_nicht_verfuegbar_Stunde1_{c}"
    )


# 6. Zielfunktion (Optional / Maximierung der stattfindenden Stunden)
# Hauptziel: es sollen so viele Stunden wie möglich stattfinden

# Weiche Zusatanforderung: Frau Schmidt arbeitet ungerne in Stunde 3.
# Wir geben diesen Variablen in der Zielfunktion ein negatives Gewicht (Strafpunkt).

# lineare Zielfunktion für Hauptziel
aktuelle_gewichte = {qp.get_variable(i).name: 1 for i in range(qp.get_num_vars())}

# Strafe einbauen: Wenn Frau Schmidt in Stunde 3 unterrichtet, bringt das 0 Punkte statt 1
for c in klassen:
    straf_variable = f"x_Stunde_3_{c}_Frau_Schmidt"
    aktuelle_gewichte[straf_variable] = 0  # Reduziert den Anreiz für den Algorithmus

print("Zielfunktion ")
print(aktuelle_gewichte)
qp.maximize(linear=aktuelle_gewichte)

print(qp.export_as_lp_string())


# Struktur überprüfen
print(f"Anzahl Variablen: {qp.get_num_vars()}")
print(f"Anzahl Nebenbedingungen: {qp.get_num_linear_constraints()}")

Binäre Variablen:
x_Stunde_1_Klasse_A_Herr_Mueller
x_Stunde_1_Klasse_A_Frau_Schmidt
x_Stunde_1_Klasse_B_Herr_Mueller
x_Stunde_1_Klasse_B_Frau_Schmidt
x_Stunde_2_Klasse_A_Herr_Mueller
x_Stunde_2_Klasse_A_Frau_Schmidt
x_Stunde_2_Klasse_B_Herr_Mueller
x_Stunde_2_Klasse_B_Frau_Schmidt
x_Stunde_3_Klasse_A_Herr_Mueller
x_Stunde_3_Klasse_A_Frau_Schmidt
x_Stunde_3_Klasse_B_Herr_Mueller
x_Stunde_3_Klasse_B_Frau_Schmidt
Nebenbedingung 1: Eine Klasse hat pro Stunde max. einen Lehrer
{'x_Stunde_1_Klasse_A_Herr_Mueller': 1, 'x_Stunde_1_Klasse_A_Frau_Schmidt': 1}
{'x_Stunde_1_Klasse_B_Herr_Mueller': 1, 'x_Stunde_1_Klasse_B_Frau_Schmidt': 1}
{'x_Stunde_2_Klasse_A_Herr_Mueller': 1, 'x_Stunde_2_Klasse_A_Frau_Schmidt': 1}
{'x_Stunde_2_Klasse_B_Herr_Mueller': 1, 'x_Stunde_2_Klasse_B_Frau_Schmidt': 1}
{'x_Stunde_3_Klasse_A_Herr_Mueller': 1, 'x_Stunde_3_Klasse_A_Frau_Schmidt': 1}
{'x_Stunde_3_Klasse_B_Herr_Mueller': 1, 'x_Stunde_3_Klasse_B_Frau_Schmidt': 1}
Nebenbedingung 2: Ein Lehrer unterrichtet pro Stu

/tmp/ipykernel_1475/2246846104.py:104: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print(qp.export_as_lp_string())


In [1]:
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver  # Benötigt pip install qiskit-algorithms

# ==========================================
# 1. MODELL AUFBAUEN (aus dem vorherigen Schritt)
# ==========================================
qp = QuadraticProgram(name="Stundenplan_Loesung")

zeitslots = ["Stunde_1", "Stunde_2", "Stunde_3"]
klassen = ["Klasse_A", "Klasse_B"]
lehrer = ["Herr_Mueller", "Frau_Schmidt"]

for t in zeitslots:
    for c in klassen:
        for tchr in lehrer:
            qp.binary_var(name=f"x_{t}_{c}_{tchr}")

# Grund-Constraints (Maximal 1 Lehrer pro Klasse, maximal 1 Klasse pro Lehrer zur selben Zeit)
for t in zeitslots:
    for c in klassen:
        qp.linear_constraint(linear={f"x_{t}_{c}_{l}": 1 for l in lehrer}, sense="<=", rhs=1, name=f"Constraint_Klasse_{c}_zur_{t}")
for t in zeitslots:
    for tchr in lehrer:
        qp.linear_constraint(linear={f"x_{t}_{c}_{tchr}": 1 for c in klassen}, sense="<=", rhs=1, name=f"Constraint_Lehrer_{tchr}_zur_{t}")

# Fachanforderungen hinzufügen
# Anforderung: Herr Müller muss mindestens 2 Stunden in Klasse A unterrichten
anforderung_stunden_mueller_a = 2
constraint_dict_mueller_a = {}
for t in zeitslots:
    constraint_dict_mueller_a[f"x_{t}_Klasse_A_Herr_Mueller"] = 1
qp.linear_constraint(linear=constraint_dict_mueller_a, sense=">=", rhs=anforderung_stunden_mueller_a, name="Mueller_A_2h")

# Anforderung: Frau Schmidt muss mindestens 1 Stunde in Klasse B unterrichten
anforderung_stunden_schmidt_b = 1
constraint_dict_schmidt_b = {}
for t in zeitslots:
    constraint_dict_schmidt_b[f"x_{t}_Klasse_B_Frau_Schmidt"] = 1
qp.linear_constraint(linear=constraint_dict_schmidt_b, sense=">=", rhs=anforderung_stunden_schmidt_b, name="Schmidt_B_1h")

# Anforderung: jede Klasse mindestens 2 Unterrichtsstunden
anforderung_stunden_klasse = 2
for c in klassen:
    constraint_dict_klasse = {}
    for t in zeitslots:
        for tchr in lehrer:
            constraint_dict_klasse[f"x_{t}_{c}_{tchr}"] = 1

    qp.linear_constraint(linear=constraint_dict_klasse, sense=">=", rhs=anforderung_stunden_klasse, name=f"Klasse_{c}_3h")

# Anforderung: Herr Müller kann in Stunde 1 KEINE Klasse unterrichten (Verfügbarkeit)
for c in klassen:
    qp.linear_constraint(
        linear={f"x_Stunde_1_{c}_Herr_Mueller": 1},
        sense="==",
        rhs=0,
        name=f"Mueller_nicht_verfuegbar_Stunde1_{c}"
    )

# Zielfunktion: Belohnung für jede gesetzte Stunde (maximieren)
# Hauptziel: es sollen so viele Stunden wie möglich stattfinden
aktuelle_gewichte = {qp.get_variable(i).name: 1 for i in range(qp.get_num_vars())}

# Weiche Zusatanforderung: Frau Schmidt arbeitet ungerne in Stunde 3.
# Wir geben diesen Variablen in der Zielfunktion ein negatives Gewicht (Strafpunkt).
for c in klassen:
    straf_variable = f"x_Stunde_3_{c}_Frau_Schmidt"
    aktuelle_gewichte[straf_variable] = 0  # Reduziert den Anreiz für den Algorithmus
qp.maximize(linear=aktuelle_gewichte)

# ==========================================
# 2. OPTIMIZER AUFSETZEN & PROBLEM LÖSEN
# ==========================================
# Wir nutzen den klassischen Eigensolver als Referenz, um die mathematische Korrektheit zu prüfen
exact_solver = NumPyMinimumEigensolver()
optimizer = MinimumEigenOptimizer(exact_solver)

# Problem an den Löser übergeben (dies konvertiert Constraints intern in QUBO-Penalties)
result = optimizer.solve(qp)

# ==========================================
# 3. ERGEBNISSE AUSWERTEN & DARSTELLEN
# ==========================================
print(f"Status der Lösung: {result.status.name}\n")

# Extrahiere die Variablen, die den Wert 1 erhalten haben
print("--- GENERIERTER STUNDENPLAN ---")
stundenplan = {t: {c: "---" for c in klassen} for t in zeitslots}

for var_name, value in result.variables_dict.items():
    if value == 1.0:
        # Variable parsen (Format: x_Stunde_X_Klasse_Y_Lehrer_Z)
        parts = var_name.split("_")
        stunde = f"{parts[1]}_{parts[2]}"  # z.B. Stunde_1
        klasse = f"{parts[3]}_{parts[4]}"  # z.B. Klasse_A
        lehrer_name = f"{parts[5]}_{parts[6]}" # z.B. Herr_Mueller

        stundenplan[stunde][klasse] = lehrer_name

# Ausgabe als Matrix
header = f"{'Zeitslot':<12} | {'Klasse A':<15} | {'Klasse B':<15}"
print(header)
print("-" * len(header))
for stunde in zeitslots:
    print(f"{stunde:<12} | {stundenplan[stunde]['Klasse_A']:<15} | {stundenplan[stunde]['Klasse_B']:<15}")

Status der Lösung: SUCCESS

--- GENERIERTER STUNDENPLAN ---
Zeitslot     | Klasse A        | Klasse B       
------------------------------------------------
Stunde_1     | ---             | Frau_Schmidt   
Stunde_2     | Herr_Mueller    | Frau_Schmidt   
Stunde_3     | Herr_Mueller    | Frau_Schmidt   


In [2]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler as Sampler # Verwende V2 Sampler

# ==========================================
# 1. PROBLEM-DEFINITION (Kurzform des Stundenplans)
# ==========================================
qp = QuadraticProgram(name="Stundenplan_QAOA")

zeitslots = ["Stunde_1", "Stunde_2"]
klassen = ["Klasse_A"]
lehrer = ["Herr_Mueller", "Frau_Schmidt"]

# Variablen erzeugen
for t in zeitslots:
    for c in klassen:
        for tchr in lehrer:
            qp.binary_var(name=f"x_{t}_{c}_{tchr}")

# Harte Einschränkung: Klasse A darf pro Stunde max. 1 Lehrer haben
for t in zeitslots:
    qp.linear_constraint(linear={f"x_{t}_Klasse_A_{l}": 1 for l in lehrer}, sense="<=", rhs=1)

# Fachanforderung: Herr Müller MUSS genau 1 Stunde halten
qp.linear_constraint(linear={f"x_{t}_Klasse_A_Herr_Mueller": 1 for t in zeitslots}, sense="==", rhs=1)

# Zielfunktion
qp.maximize(linear={qp.get_variable(i).name: 1 for i in range(qp.get_num_vars())})

# ==========================================
# 2. QUANTEN-ALGORITHMUS (QAOA) KONFIGURIEREN
# ==========================================
# Der Sampler V2 wird hier initialisiert
sampler = Sampler()

# COBYLA Optimizer
classical_optimizer = COBYLA(maxiter=50)

# QAOA initialisieren
qaoa_solver = QAOA(sampler=sampler, optimizer=classical_optimizer, reps=1)

# Den Quanten-Löser in das Optimization Framework einbetten
qaoa_optimizer = MinimumEigenOptimizer(qaoa_solver)

# ==========================================
# 3. AUSFÜHRUNG & AUSWERTUNG
# ==========================================
print("Starte QAOA-Optimierungsschleife...")
result = qaoa_optimizer.solve(qp)

print(f"\nStatus der Quanten-Lösung: {result.status.name}")

# Stundenplan ausgeben
print("\n--- DURCH QAOA ERMITTELTER STUNDENPLAN ---")
for var_name, value in result.variables_dict.items():
    if value == 1.0:
        print(f"Aktiviert: {var_name}")

Starte QAOA-Optimierungsschleife...


/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.13/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])



Status der Quanten-Lösung: SUCCESS

--- DURCH QAOA ERMITTELTER STUNDENPLAN ---
Aktiviert: x_Stunde_1_Klasse_A_Frau_Schmidt
Aktiviert: x_Stunde_2_Klasse_A_Herr_Mueller


In [5]:
!pip install --upgrade qiskit qiskit-algorithms